In [ ]:
## Motivation behind running this reference database to model cross-over prediction is 
# to see if we are confident enough to use model trained on full length amplicon while
# classification of shorter region amplicons like V4, V3-V4
## As claimed by qiime2 developers, sklearn classifiers trained on full length data can be used with
# classificaiton of other shorter regions without much difference in results

import os
from os.path import join, basename, exists, expandvars, dirname, splitext
from pathlib import Path
from joblib import Parallel, delayed
from src.extract_reads import cutter
import src.utils

In [ ]:
## set path vars
n_cpus = os.cpu_count() - 1
n_jobs = 1 # number_of_jobs to run in parallel
analysis_name = "database-crossover"
project_dir = expandvars("/home/kardokhk/taxonova_bench")
conda_dir = join(project_dir, "conda")
scripts_dir = join(project_dir, f"scripts/mock-community/src")
reference_database_dir = join(project_dir, f"benchmark/data/ref_dbs")
taxonova_model_dir = join(project_dir, "models")
results_dir = join(project_dir, f"benchmark/data", analysis_name)


In [5]:
# amplicon regions
v_regions = ['V4', 'V3-V4', 'V1-V9']

# taxonova input models
taxonova_models = {
    'V4': join(taxonova_model_dir, 'my_gg2_v4_model.pt'),
    'V3-V4': join(taxonova_model_dir, 'my_gg2_v3v4_model.pt'),
    'V1-V9': join(taxonova_model_dir, 'my_gg2_full_model.pt'),
}

# input reference seqs
ref_seq = join(results_dir, 'input_data', 'gg2_backbone.fasta')

In [6]:
## Extract amplicon region based on primer-set
v_primers = src.utils.get_vregion_primers()
v_length = src.utils.get_vregion_lengths()

# check dir
outdir = join(results_dir, 'extracted_regions')
src.utils.check_dir(outdir)

# cut reads and keep only entries with both sequence and taxonomy information
out_seqs_file = {}
for v_region in v_regions:
    out_seqs_file[v_region] = {}
    # out_files
    seqs_out = join(outdir, f'{v_region}_seqs.fasta')
    if not Path(seqs_out).exists():
        # cut reads
        cutter(
            sequences = ref_seq, 
            f_primer = v_primers[v_region]['f'],
            r_primer = v_primers[v_region]['r'], 
            min_length = v_length[v_region]['min'], 
            max_length = v_length[v_region]['max'],
            n_jobs = n_cpus,
            output_path = seqs_out)
        out_seqs_file[v_region] = seqs_out
    else:
        out_seqs_file[v_region] = seqs_out

out_seqs_file

{'V4': '/home/shriram/Desktop/OUTDIR/Kardokh_UCC/benchmark/data/database-crossover/extracted_regions/V4_seqs.fasta',
 'V3-V4': '/home/shriram/Desktop/OUTDIR/Kardokh_UCC/benchmark/data/database-crossover/extracted_regions/V3-V4_seqs.fasta',
 'V1-V9': '/home/shriram/Desktop/OUTDIR/Kardokh_UCC/benchmark/data/database-crossover/extracted_regions/V1-V9_seqs.fasta'}

In [ ]:
## Classify using taxonova
outdir = join(results_dir, 'classify')
src.utils.check_dir(outdir)

# cut reads and keep only entries with both sequence and taxonomy information
commands = []
out_results_file = {}
for v_region in v_regions:
    out_results_file[v_region] = {}
    other_regions = v_regions
    for other_region in other_regions:
        out_results_file[v_region][other_region] = {}
        # out_files
        raw_results = join(outdir, f'{v_region}_{other_region}_raw.txt')
        final_results = join(outdir, f'{v_region}_{other_region}_final.txt')
        
        if not Path(raw_results).exists() and not Path(final_results).exists():
                # assemble command
            cmd = (
                f"taxonova classification "
                    f"--confidence_threshold 0.50 "
                    f"--input {out_seqs_file[other_region]} "
                    f"--model_input {taxonova_models[v_region]} "
                    f"--output {raw_results} && "
                f"python {scripts_dir}/process_taxonova_output.py "
                    f"{raw_results} {final_results} False "
            )
            commands.append(cmd)

# run each command
Parallel(n_jobs=1) (delayed(os.system)(command) for command in list(commands))



9